<a href="https://colab.research.google.com/github/JAVERIAADIL/Learning-GPU-infrastructure/blob/module1/VectorMul.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [13]:
import cupy as cp
from numba import cuda

In [8]:
N= 3
vector_1 = [[4,5,6],[7,8,9],[1,2,3]]
vector_2 = [[1,2,3],[4,5,6],[7,8,9]]
result = [[0,0,0],[0,0,0],[0,0,0]]

For size of vectors and threads
Thread cannot be more than 1024 in a block that is its hardware limit so we cannot give dimension of block as 512 * 512 as it will be 262144 so we can give like 16 * 16 etc and for grid we can calculate it by N/ block dimension like 16 etc so if N which is one dimension of vector is 512  we have to benchmark to scale grids on basis of N

In [14]:
N = 128
vector_1 = cp.random.randn(128,128)
vector_2 = cp.random.randn(128,128)
result = cp.zeros((128,128))

In [3]:
# cupy_vector1 = cp.array(vector_1).flatten()
# cupy_vector2 = cp.array(vector_2).flatten()
# c_result = cp.array(result).flatten()

In [15]:
cupy_vector1 = cp.array(vector_1)
cupy_vector2 = cp.array(vector_2)
c_result = cp.array(result)

In memory vectors are always stored as 1d array and above given vectors are 2d array so we can flatten it as in formula later we can do calculation as flat 1d array using N as stride but we can also calculate it as 2d array in cupy/numoy and beneath they do same 1d calculation like in cuda c and we convert our vector into cupy array as normal list is like a python object in memory while gpu needs to know size type and location of array in memory so we convert it

In [16]:
d_vector1 = cuda.to_device(cupy_vector1)
d_vector2 = cuda.to_device(cupy_vector2)
d_result = cuda.to_device(c_result)

In [5]:
@cuda.jit
def vectorMul(d_vector1, d_vector2, d_result_arg):
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y # here row gives one complete row and same for column
    column = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if ( row < N and column < N): # since one thread is assigned to calculate one value so for calcultion every thread has access to 1 complete row and 1 complete colmn to multiple each row value to each column value and then add
      sum = 0
      for K in range(N): # that is why we use loop which is for each thread rather than cpu where we use 2 loops for rows and columns
        sum += d_vector1[row * N + K] * d_vector2[K * N + column] # here N is used as stride as we have flat array like this [3,4,5,6,7,8] so by N we know how reach that specific row in a array by * row index with N which is total value in 1d like for 3*3 array N is 3
      d_result_arg[row * N + column] = sum # single thread is assigned to calculate d_result and to place its value in correct position and here also N is used as stride

In [17]:
#same code but with 2d calculation
@cuda.jit
def vectorMul(d_vector1, d_vector2, d_result_arg):
    row = cuda.blockIdx.y * cuda.blockDim.y + cuda.threadIdx.y
    column = cuda.blockIdx.x * cuda.blockDim.x + cuda.threadIdx.x
    if ( row < N and column < N):
      sum = 0
      for K in range(N):
        sum += d_vector1[row][K] * d_vector2[K] [column]
      d_result_arg[row][column] = sum

In [18]:
block = (32, 32)
grid = (N // 32, N // 32)


start_event = cuda.event()
end_event = cuda.event()

start_event.record()
vectorMul[grid, block](d_vector1, d_vector2, d_result) #in [] always give uper level first like if grid and block then give grid , block if thread block give block, thread
end_event.record()
end_event.synchronize()

print ( f" GPU Average: {cuda.event_elapsed_time(start_event, end_event):.3f} ms")

result = d_result.copy_to_host()

print(result)

 GPU Average: 162.691 ms
[[  3.37284297  -2.16091831  -4.50819918 ...   1.05222413   4.97068737
   -5.24725619]
 [ -9.20853851  -6.80227586  -1.03323185 ...   3.87980559   0.99505211
   10.92875964]
 [ -3.26378474  -2.7612725    5.59912577 ...   7.9615983  -17.05987009
   -7.67833408]
 ...
 [ -5.76935545  11.58341148   1.6064101  ... -12.4123774  -15.50181121
   21.81561884]
 [  7.15648093   3.83988841   1.64228061 ...  13.86121028   3.5188147
    5.23063124]
 [  5.79497026  -1.65260646  11.44825619 ...   8.47614135 -12.41664462
    0.4468308 ]]


/usr/local/lib/python3.12/dist-packages/numba_cuda/numba/cuda/dispatcher.py:696: NumbaPerformanceWarning: Grid size 16 will likely result in GPU under-utilization due to low occupancy.
  warn(errors.NumbaPerformanceWarning(msg))
